### LLM EvaluationHow to measure LLM output quality: intrinsic language-modeling metrics (cross entropy, perplexity), generation-quality metrics (BLEU, ROUGE, BERTScore, LLM-as-judge, Pass@k), RAG/retrieval metrics (Precision@k, Recall@k, MRR, NDCG, RAGAS), and the eval libraries used in practice. Classification/regression metrics (confusion matrix, ROC-AUC, MAE/RMSE, calibration) are general ML metrics, covered in `fundamentals/eval-metrics.ipynb` instead.

#### Generation quality metrics: concept, worked by hand

#### Concept note: LLM generation quality — Perplexity, BLEU, ROUGE, BERTScore, LLM-as-judge, Pass@k, Human eval

Researched current usage (2026): no single metric captures LLM quality —
perplexity measures fluency, BLEU/ROUGE measure surface overlap, BERTScore
measures meaning, LLM-as-judge (G-Eval) correlates best with human judgment
(0.40-0.60 range) but costs an extra LLM call per evaluation.

0. Perplexity = exp(cross-entropy loss) — how "surprised" the model is by
   the true next tokens, on average.
   Toy 3-token sequence, model's assigned probability to each true token:
   [0.5, 0.25, 0.8]
   CE = -(1/3)[ln(0.5) + ln(0.25) + ln(0.8)] = -(1/3)(-0.693-1.386-0.223)
      = -(1/3)(-2.302) = 0.767
   Perplexity = exp(0.767) ≈ 2.154 — interpreted as "on average, as confused
   as choosing uniformly among ~2.15 options per token." Lower is better;
   perplexity=1 means perfectly confident and correct every time.

1. BLEU (simplified unigram version — real BLEU uses geometric mean of 1-4
   gram precision + a brevity penalty):
   reference: "the cat sat on the mat" (6 tokens)
   candidate: "the cat is on the mat" (6 tokens)
   matching unigrams (clipped to reference counts): the(2/2), cat(1/1),
   is(0, not in reference), on(1/1), mat(1/1) → 5 matches out of 6 candidate tokens
   precision₁ = 5/6 ≈ 0.833
   Brevity penalty = 1 (candidate length = reference length, no penalty)
   BLEU (simplified) ≈ 0.833
   Limitation: "is" got zero credit even though it's a reasonable word choice
   here — BLEU only rewards exact token matches, no notion of meaning.

2. ROUGE-L — based on Longest Common Subsequence (LCS), standard for
   summarization since it rewards word ORDER, not just presence.
   Same reference/candidate as above. LCS = "the cat on the mat" (skips
   "sat" from reference, "is" from candidate) → LCS length = 5
   ROUGE-L recall = LCS/len(reference) = 5/6 ≈ 0.833
   ROUGE-L precision = LCS/len(candidate) = 5/6 ≈ 0.833
   (equal here since both sequences are the same length)

3. BERTScore — token-level cosine similarity between contextual embeddings
   of candidate and reference (not exact string match), greedily matched,
   then averaged. Can't hand-compute without real embeddings, but the point
   it fixes: it would give "is" partial credit for being semantically in the
   neighborhood of "sat" (both are common verbs in a similar sentence
   position), where BLEU/ROUGE give it exactly zero.

4. LLM-as-judge (G-Eval framework) — use a second LLM call with a rubric and
   chain-of-thought reasoning to score the output on custom criteria, since
   open-ended generation has no single "correct" reference to overlap-score
   against. Reported to reach 0.40-0.60 correlation with human scores,
   well above surface-overlap metrics like BLEU/ROUGE on open-ended tasks.
   Mechanism: same pattern as this project's structured-output extraction —
   the "judge" call returns a structured score (e.g. JSON schema with a
   1-5 rating + reasoning field) rather than free text, for parseable, auditable results.

5. Pass@k — code generation: probability that AT LEAST ONE of k sampled
   generations passes the test suite, given n total samples were generated
   and c of them were actually correct (unbiased estimator, doesn't require
   generating exactly k every time):
   pass@k = 1 - C(n-c, k) / C(n, k)
   Toy: generated n=5 samples, c=2 were correct, budget k=1:
   pass@1 = 1 - C(3,1)/C(5,1) = 1 - 3/5 = 0.4

6. Human eval / win-rate — pairwise preference (show two model outputs,
   which is better?) aggregated into a win-rate or Elo-style rating, the
   same ranking mechanism chess/chatbot arenas use. Most expensive and
   slowest to collect, but the ground truth every automatic metric above is
   ultimately trying to approximate — used for safety-critical or highly
   subjective tasks where no automatic proxy is trusted alone.


In [ ]:
import numpy as np
import math
from collections import Counter

# perplexity
token_probs = [0.5, 0.25, 0.8]
cross_entropy = -np.mean([np.log(p) for p in token_probs])
perplexity = np.exp(cross_entropy)
print("perplexity:", perplexity)

# simplified BLEU-1 (unigram precision + brevity penalty; real BLEU uses 1-4 gram geometric mean)
reference = "the cat sat on the mat".split()
candidate = "the cat is on the mat".split()

ref_counts = Counter(reference)
cand_counts = Counter(candidate)
matches = sum(min(cand_counts[w], ref_counts[w]) for w in cand_counts)
precision_1 = matches / len(candidate)
brevity_penalty = 1.0 if len(candidate) >= len(reference) else math.exp(1 - len(reference) / len(candidate))
bleu_simplified = precision_1 * brevity_penalty
print("BLEU (simplified unigram):", bleu_simplified)

# ROUGE-L via LCS
def lcs_length(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[-1][-1]

lcs = lcs_length(reference, candidate)
rouge_l_recall = lcs / len(reference)
rouge_l_precision = lcs / len(candidate)
print(f"ROUGE-L: recall={rouge_l_recall:.3f} precision={rouge_l_precision:.3f}")

# pass@k
n, c, k = 5, 2, 1
pass_at_k = 1 - math.comb(n - c, k) / math.comb(n, k)
print("pass@1:", pass_at_k)

#### Generation quality metrics: real implementations (nltk BLEU, rouge-score, bert-score, LLM-as-judge)

#### Intrinsic Evaluations

Language modeling efficiency, the mathematical divergence between the learned probability distribution and the true training-data distribution.

#### 1. Entropy

Entropy = average information (bits) a token carries. Higher entropy → harder to predict what comes next; lower entropy → more predictable.

Analogy: if you can perfectly predict what comes next, it carries no new information. Entropy is highest when outcomes are equally likely, lowest when deterministic.

- Highly structured domains (HTML, SQL, JSON): lower entropy.
- Casual conversation or creative text: higher entropy.

#### 2. Cross Entropy

Training an LLM means learning the distribution of the training data. Cross entropy on a dataset measures how difficult it is for the model to predict what comes next.

If $P$ is the true data distribution and $Q$ the model's learned distribution:

$$H(P, Q) = H(P) + D_{KL}(P||Q)$$

$D_{KL}$ measures how far the learned distribution diverges from the true one, the expected extra bits needed to encode samples from $P$ using $Q$ instead:

$$D_{KL}(P \| Q) = \sum_{x} P(x) \cdot \log \left(\frac{P(x)}{Q(x)}\right)$$

If $P=Q$: $\frac{P(x)}{Q(x)}=1 \Rightarrow \log(\cdot)=0 \Rightarrow D_{KL}(P\|Q)=0$.

A model's cross-entropy can never go below the dataset's inherent entropy $H(P)$, the theoretical lower bound.

Cross-entropy and KL divergence are both asymmetric (not symmetric in $P, Q$).

---

#### 3. Perplexity

Exponentiated cross entropy, measures the model's absolute uncertainty in predicting the next token. In bits (base 2):

$$PPL(P, Q) = 2^{H(P, Q)}$$

Or with natural log (base $e$):
$$PPL(P, Q) = e^{H(P, Q)}$$

Lower perplexity = model assigns higher probability to the true token sequence = less uncertainty. Higher perplexity = more uncertainty.

---

#### N-Gram Ovelap

#### 1. BLEU 

BLEU (Bilingual Evaluation Understudy) measures how many n-grams in the generated text appear in the reference (precision-focused). It is a metric primarily used to evaluate machine translation systems and produces a score between 0 and 1 (often reported as 0 to 100).

An 'n-gram' is just a way of describing 'n' consecutive words in a sentence. For instance, in the sentence "The cat is sleeping" we have bigrams (2-gram) as: "The cat", "cat is", "is sleeping". Also note that the words in an n-gram are taken in order, so jumbling up won't generate valid n-grams.


#### 2. ROUGE

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) is the recall-focused counterpart to BLEU, commonly used for summarization, measures how much of the reference text's content is captured by the generated summary. Key variants:

 - ROUGE-N: Measures n-gram overlap (similar to BLEU but recall-based).
 - ROUGE-L: Uses the longest common subsequence (LCS) to capture sentence-level structure similarity.
 - ROUGE-S: Measures skip-bigram overlap (pairs of words in order but not necessarily adjacent).
 - Emphasizes coverage of important content from the reference summary.

Because summarization aims to include as much essential information as possible (rather than avoiding extra words), recall is often more important than precision in that context.

#### 3. Pairwise comparative evaluation and Elo ratings

Pairwise comparative evaluation: present two outputs for the same prompt to a human or AI judge, ask which is better (or tied). Outputs can be from different models, or different responses of the same model.

Elo rating (originally for ranking chess players by relative skill, not win/loss counts) adapts directly: two LLM responses to the same prompt are a "match," judged by a human or automated system. A lower-rated model that wins a comparison gains rating points; over thousands of comparisons, ratings converge to reflect relative performance.

Limitation: Elo assumes transitivity (A beats B, B beats C ⟹ A beats C), which doesn't always hold across task types, and is sensitive to prompt/evaluator quality and diversity.

---

In [ ]:
source_text = """
The Amazon rainforest, often referred to as the "lungs of the Earth," produces
approximately 20 percent of the world's oxygen. Spanning over 5.5 million
square kilometers across nine countries in South America, it is the largest
tropical rainforest on the planet. The Amazon is home to an estimated 10
percent of all species on Earth, including over 40,000 plant species, 1,300
bird species, and 3,000 types of fish. Deforestation, primarily driven by
cattle ranching and soybean farming, poses the greatest threat to this
ecosystem. Between 2001 and 2020, the Amazon lost approximately 10 percent
of its forest cover. Scientists warn that continued deforestation could push
the rainforest past a tipping point, transforming large portions into savanna
and releasing billions of tons of stored carbon dioxide into the atmosphere.
"""

reference_summary = (
    "The Amazon rainforest is the world's largest tropical forest, producing "
    "about 20% of global oxygen and hosting 10% of Earth's species. "
    "Deforestation from agriculture threatens to push it past a tipping point, "
    "potentially converting it to savanna and releasing massive carbon emissions."
)

# Candidate A: a good summary that captures key points in different words
candidate_a = (
    "Spanning 5.5 million square kilometers, the Amazon is Earth's biggest "
    "tropical rainforest and a critical oxygen source. It harbors extraordinary "
    "biodiversity, but agricultural deforestation, mainly cattle and soy, "
    "risks triggering an irreversible shift to savanna, which would unleash "
    "vast carbon stores."
)

# Candidate B: a flawed summary, vague, misses key facts, adds a hallucination
candidate_b = (
    "The Amazon is a large forest in South America. It has many trees and "
    "animals. Some people cut down trees there. The forest was designated a "
    "UNESCO World Heritage Site in 2005."  # <-- hallucinated fact
)

#### BLEU Score

BLEU measures how many n-grams in the generated text appear in the reference (precision-focused).



In [2]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


def compute_bleu(candidate, reference):
    ref_tokens = reference.lower().split()
    cand_tokens = candidate.lower().split()

    # smoothing function
    smoothie = SmoothingFunction().method1
    return sentence_bleu([ref_tokens], cand_tokens, smoothing_function=smoothie)

bleu_a = compute_bleu(candidate_a, reference_summary)
bleu_b = compute_bleu(candidate_b, reference_summary)

print(f"BLEU - Candidate A: {bleu_a:.4f}")
print(f"BLEU - Candidate B: {bleu_b:.4f}")

BLEU - Candidate A: 0.0147
BLEU - Candidate B: 0.0123


#### ROUGE Score

ROUGE measures recall: how much of the reference content is captured by the candidate? This is especially relevant for summarization.



In [3]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def compute_rouge(candidate, reference):
    scores = scorer.score(reference, candidate)
    return {key: round(val.fmeasure, 4) for key, val in scores.items()}

rouge_a = compute_rouge(candidate_a, reference_summary)
rouge_b = compute_rouge(candidate_b, reference_summary)

print(f"ROUGE, Candidate A: {rouge_a}")
print(f"ROUGE, Candidate B: {rouge_b}")

ROUGE — Candidate A: {'rouge1': 0.3908, 'rouge2': 0.0706, 'rougeL': 0.2529}
ROUGE — Candidate B: {'rouge1': 0.2368, 'rouge2': 0.027, 'rougeL': 0.1579}


#### BERT Score

Measures semantic similarity between reference and candidate, catches paraphrasing/wording differences that n-gram overlap misses.

- Precision: for each candidate token, max cosine similarity with all reference tokens, averaged across candidate tokens.
- Recall: symmetric, for each reference token, max similarity with all candidate tokens, averaged across reference tokens.
- F1: harmonic mean of precision and recall.

In [4]:
from bert_score import score as bert_score

def compute_bertscore(candidates, references):
    P, R, F1 = bert_score(candidates, references, model_type="roberta-large", lang="en", verbose=False)
    return P.tolist(), R.tolist(), F1.tolist()

# Compute scores
precision_scores, recall_scores, f1_scores = compute_bertscore(
    [candidate_a, candidate_b],
    [reference_summary, reference_summary]
)

# Candidate A
print(f"Candidate A, Precision: {precision_scores[0]:.4f}")
print(f"Candidate A, Recall:    {recall_scores[0]:.4f}")
print(f"Candidate A, F1:        {f1_scores[0]:.4f}")

print()

# Candidate B
print(f"Candidate B, Precision: {precision_scores[1]:.4f}")
print(f"Candidate B, Recall:    {recall_scores[1]:.4f}")
print(f"Candidate B, F1:        {f1_scores[1]:.4f}")

/Users/monusingh/work-share/code-blogs-articles/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 860.99it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loadi

Candidate A — Precision: 0.8948
Candidate A — Recall:    0.9078
Candidate A — F1:        0.9012

Candidate B — Precision: 0.8875
Candidate B — Recall:    0.8474
Candidate B — F1:        0.8670


In [4]:
from dotenv import load_dotenv
import os

load_dotenv()  # Loads from .env in the project root
api_key = os.environ.get("OPENROUTER_API_KEY")

if not api_key:
    raise ValueError("OPENROUTER_API_KEY not found in .env file")

os.environ["OPENROUTER_API_KEY"] = api_key

In [6]:
import litellm
import json

client = litellm.LiteLLM()

JUDGE_PROMPT = """You are an expert evaluator assessing the quality of text summaries.

Given a source text, a reference summary (gold standard), and a candidate summary evaluate
the candidate on three criteria.

For each criterion, assign a score of 0 (fail) to 1 (pass) and provide a
brief justification.

Compare against both the source text AND the reference summary.

**Criteria:**
1. **Accuracy**, Are all claims in the candidate summary factually supported by the source?
   Any hallucinated or fabricated information => accuracy=0.
2. **Completeness**, Does the candidate summary capture ALL the key points of the source given the reference?
   Any missing key points => completeness=0.
3. **Conciseness**, Is the candidate summary brief? Or is it too verbose compared to reference summary?
   Significantly extra verbosity in candidate, given the reference => conciseness=0.

### Response format
Respond with ONLY a valid JSON object (no markdown fences, no commentary):
{
  "accuracy":     {"score": <0 or 1>, "justification": "<1-2 sentences>"},
  "completeness": {"score": <0 or 1>, "justification": "<1-2 sentences>"},
  "conciseness":  {"score": <0 or 1>, "justification": "<1-2 sentences>"}
}
"""

DATA_PROMPT = """\
SOURCE TEXT:
{source}

REFERENCE SUMMARY:
{reference}

CANDIDATE SUMMARY:
{candidate}
"""

def llm_judge(source, reference, candidate, max_retries=2):

    for attempt in range(max_retries + 1):
        resp = client.chat.completions.create(
            model="openrouter/openai/gpt-4o-2024-08-06",
            messages=[
                {"role": "system", "content": JUDGE_PROMPT},
                {"role": "user", "content": DATA_PROMPT.format(
                    source=source, reference=reference, candidate=candidate
                )},
            ],
            temperature=0.00,
            max_tokens=1000
        )

        text = resp.choices[0].message.content.strip()

        # Robust JSON extraction: handle markdown fences (if suppose the model adds them)
        if text.startswith("```"):
            text = text.split("\n", 1)[1].rsplit("```", 1)[0].strip()

        # Hard-validate JSON; retry once if invalid
        try:
            data = json.loads(text)
            # minimal schema check
            for k in ("accuracy", "completeness", "conciseness"):
                assert k in data and "score" in data[k] and "justification" in data[k]
                assert data[k]["score"] in (0, 1)
            return data
        except Exception:
            if attempt == max_retries:
                # return raw for debugging if it keeps failing
                return {"error": "invalid_json", "raw": text}

# overall score
def overall_score(data):
    scores = [data[k]["score"] for k in ("accuracy", "completeness", "conciseness")]
    average_score = sum(scores) / len(scores)
    print(f"\nAverage score: {average_score}")
    if data["accuracy"]["score"] == 1 and average_score >= 0.65:
        return "PASS"
    else:
        return "FAIL"


print("=" * 60)
print("JUDGE EVALUATION, CANDIDATE A")
print("=" * 60)
data1 = llm_judge(source_text, reference_summary, candidate_a)
print(json.dumps(data1, indent=2))
print ("\n Overall:")
print(overall_score(data1))


print("\n" + "=" * 60)
print("JUDGE EVALUATION, CANDIDATE B")
print("=" * 60)
data2 = llm_judge(source_text, reference_summary, candidate_b)
print(json.dumps(data2, indent=2))
print ("\n Overall:")
print(overall_score(data2))

JUDGE EVALUATION — CANDIDATE A
{
  "accuracy": {
    "score": 1,
    "justification": "The candidate summary accurately reflects the source text, including the size of the Amazon, its role in oxygen production, biodiversity, and the threat of deforestation."
  },
  "completeness": {
    "score": 1,
    "justification": "The candidate summary captures all key points from the source text, including the size, biodiversity, deforestation causes, and potential consequences."
  },
  "conciseness": {
    "score": 1,
    "justification": "The candidate summary is concise and comparable in length to the reference summary, without unnecessary verbosity."
  }
}

 Overall:

Average score: 1.0
PASS

JUDGE EVALUATION — CANDIDATE B
{
  "accuracy": {
    "score": 0,
    "justification": "The candidate summary inaccurately claims the Amazon was designated a UNESCO World Heritage Site in 2005, which is not supported by the source text."
  },
  "completeness": {
    "score": 0,
    "justification": "The 

#### The Eval Pipeline as a Process, Not Just Metrics

The metrics above (entropy, perplexity, BLEU/ROUGE/BERTScore) are the *what to measure*; this is the *how it actually runs end to end* for a generative product feature, from before launch through production, since open-ended outputs make simple accuracy insufficient.

1. Build a golden dataset first: 100–500 representative prompts spanning common use cases, edge cases, and adversarial/red-teaming inputs, with ground-truth or high-quality human reference answers.
2. Automate the eval pipeline into CI/CD: run LLM-as-a-judge plus deterministic checks on every prompt iteration, fine-tune, or system-prompt change, before it ships.
3. Log production telemetry at runtime: latency, token cost, fallback rates, and explicit user feedback signals (thumbs up/down, copy-to-clipboard, retry frequency).
4. Close the loop post-deployment: sample low-rated production logs and unexpected outputs back into the golden dataset, so the benchmark keeps improving rather than staying frozen at launch.

#### RAG / retrieval metrics: concept, worked by hand

#### Concept note: RAG / retrieval metrics — Precision@k, Recall@k, MRR, NDCG, Hit Rate, RAGAS

Researched current usage (2026) before writing this — retrieval metrics from
classic IR, RAG-specific ones from the RAGAS framework's component breakdown.

0. Setup: query "fraud detection dataset", top-5 retrieved docs ranked by the
   system, relevance labels (1=relevant, 0=not): [0, 1, 0, 1, 1]
   (first result irrelevant, then hits at rank 2, 4, 5)
   Total relevant docs that exist in the whole corpus for this query = 4
   (so 1 relevant doc exists but never made it into the top 5)

1. Precision@k = relevant retrieved / k — of what you retrieved, how much
   was actually relevant.
   Precision@5 = 3/5 = 0.6

2. Recall@k = relevant retrieved / total relevant that exist — of everything
   relevant out there, how much did you surface.
   Recall@5 = 3/4 = 0.75

3. MRR (Mean Reciprocal Rank) = 1/rank of the FIRST relevant result, averaged
   across queries — cares only about how fast you hit something useful.
   This query: first relevant is at rank 2 → reciprocal rank = 1/2 = 0.5
   (average this across many queries for the "mean" part — a single query
   only gives one reciprocal rank)

4. NDCG@k (Normalized Discounted Cumulative Gain) — like precision@k, but
   weights hits by position (an early hit counts more than a late one) and
   normalizes against the best-possible ordering.
   DCG@5 = Σ rel_i / log2(i+1) for i=1..5, rel=[0,1,0,1,1]
   = 0/log2(2) + 1/log2(3) + 0/log2(4) + 1/log2(5) + 1/log2(6)
   ≈ 0 + 0.631 + 0 + 0.431 + 0.387 = 1.449
   IDCG@5 (ideal: all 3 relevant docs ranked first, rel=[1,1,1,0,0]):
   = 1/log2(2) + 1/log2(3) + 1/log2(4) = 1 + 0.631 + 0.5 = 2.131
   NDCG@5 = DCG/IDCG = 1.449/2.131 ≈ 0.680

5. Hit Rate = fraction of QUERIES (not documents) where at least one relevant
   doc appears in top-k — a per-query yes/no, averaged.
   Toy across 3 queries: query1=hit(1), query2=miss(0), query3=hit(1) →
   Hit Rate = 2/3 ≈ 0.667

6. RAGAS-style component metrics (the standard RAG evaluation breakdown,
   splits "is the RAG system good" into retrieval quality vs. generation quality):
   - Context Precision: are the retrieved chunks actually relevant, and
     ranked with the relevant ones first? (retrieval-side precision, same
     idea as precision@k applied specifically to a RAG pipeline's context)
   - Context Recall: does the retrieved context contain everything needed to
     produce the ground-truth answer? (retrieval-side recall)
   - Faithfulness: does the generated answer's claims actually appear in the
     retrieved context, or did the model add unsupported content? — this is
     exactly the extraction-faithfulness spot-audit used in the fraud project
     (check the LLM's output against the source text it was given), just
     applied to a full generated answer instead of a structured field.
   - Answer Relevancy: does the answer actually address the question asked
     (checked by generating synthetic questions FROM the answer, then
     comparing those back to the original question — if they diverge, the
     answer likely went off-topic even if faithful to the context).


In [ ]:
import numpy as np

relevance = np.array([0, 1, 0, 1, 1])  # top-5 retrieved docs, 1=relevant
total_relevant = 4

precision_at_5 = relevance.sum() / len(relevance)
recall_at_5 = relevance.sum() / total_relevant
first_hit_rank = np.argmax(relevance == 1) + 1  # first index where relevant=1, 1-indexed
mrr_single_query = 1 / first_hit_rank

dcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(relevance))  # i is 0-indexed, +2 matches rank i+1 in log2(rank+1)
ideal_relevance = sorted(relevance, reverse=True)
idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal_relevance))
ndcg_at_5 = dcg / idcg

print(f"Precision@5={precision_at_5:.3f}  Recall@5={recall_at_5:.3f}  MRR={mrr_single_query:.3f}  NDCG@5={ndcg_at_5:.3f}")

# hit rate across multiple queries (toy: 3 queries, each either hit(1) or miss(0) in top-k)
hits_per_query = [1, 0, 1]
hit_rate = sum(hits_per_query) / len(hits_per_query)
print("Hit Rate:", hit_rate)

#### Eval libraries used in practice

#### Concept note: Eval libraries used in practice — BERTScore, RAGAS, DeepEval, Arize Phoenix, Promptfoo

Researched current (2026) tooling — these are what teams actually reach for
in production instead of hand-rolling the metrics above; each has a
different scope, not interchangeable.

0. bert-score — the real, standard implementation of BERTScore (the manual
   version earlier only illustrated the concept).
   Install: `pip install bert-score`

1. RAGAS — the standard framework for RAG-specific metrics; its metric set
   IS the faithfulness/answer-relevancy/context-precision/context-recall
   breakdown covered in the retrieval note above. Uses an LLM as judge
   internally for the semantic metrics, runs against a dataset of
   {question, answer, contexts, ground_truth} rows.
   Install: `pip install ragas datasets langchain-openai`

2. DeepEval — pytest-style unit testing for LLM apps: write assertions like
   a normal test suite, but the assertion is "is this response faithful /
   relevant / non-hallucinated," scored via LLM-as-judge (implements G-Eval
   directly). Fits naturally into CI — fails a build if quality drops below
   a threshold, same mental model as a normal test suite.
   Install: `pip install deepeval openai python-dotenv`

3. Arize Phoenix — observability + evaluation combined: traces every LLM
   call (OpenTelemetry-based), then runs evaluators (hallucination, QA
   correctness, relevance) over the traced spans, logs results back to a
   local trace UI. Different scope than RAGAS/DeepEval — built for
   monitoring a LIVE system over time, not scoring one offline eval run.
   Install: `pip install arize-phoenix`

4. Promptfoo — CLI-first, config-driven (YAML), not a Python library at all
   (Node-based, run via `npx`). Define prompts + models + test cases +
   assertions in `promptfooconfig.yaml`, run `promptfoo eval` to test every
   prompt×model×input combination in a matrix — best fit for comparing
   multiple models/prompt versions side by side, or CI regression testing
   across prompt changes.
   No pip install — run directly: `npx promptfoo@latest init`


In [ ]:
# These require installing the packages first (not run here — this env doesn't
# have them). Illustrative usage, one block per tool.

# --- bert-score --- (pip install bert-score)
# from bert_score import score
# candidates = ["the cat is on the mat"]
# references = ["the cat sat on the mat"]
# P, R, F1 = score(candidates, references, lang="en", rescale_with_baseline=True)
# print(f"precision={P.item():.3f} recall={R.item():.3f} f1={F1.item():.3f}")

# --- RAGAS --- (pip install ragas datasets langchain-openai)
# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
# from datasets import Dataset
#
# eval_dataset = Dataset.from_dict({
#     "question": ["What typology involves a fake grandchild asking for bail money?"],
#     "answer": ["Family emergency scam"],
#     "contexts": [["Family emergency scam: scammer poses as a relative, often a grandchild, claiming to be arrested..."]],
#     "ground_truth": ["Family emergency scam"],
# })
# result = evaluate(eval_dataset, metrics=[faithfulness, answer_relevancy, context_precision, context_recall])
# print(result)

# --- DeepEval --- (pip install deepeval openai python-dotenv)
# from deepeval import assert_test
# from deepeval.metrics import GEval
# from deepeval.test_case import LLMTestCase, LLMTestCaseParams
#
# correctness_metric = GEval(
#     name="Correctness",
#     criteria="Does the actual output correctly identify the fraud typology described in the input?",
#     evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
#     threshold=0.7,
# )
# test_case = LLMTestCase(
#     input="Scammer posed as IRS agent demanding wire transfer",
#     actual_output="government_impersonation_scam",
# )
# assert_test(test_case, [correctness_metric])

# --- Arize Phoenix --- (pip install arize-phoenix)
# import phoenix as px
# px.launch_app()   # starts local trace UI at http://localhost:6006
# # instrument your LLM calls (e.g. via OpenInference auto-instrumentation),
# # then run evaluators over the collected spans and log results back:
# # from phoenix.evals import HallucinationEvaluator, run_evals
# # run_evals(dataframe=spans_df, evaluators=[HallucinationEvaluator(model)], provide_explanation=True)

print("See commented blocks above for usage — install each package to actually run them.")

Promptfoo isn't Python — its "code" is a YAML config, run via `npx promptfoo@latest eval`:

```yaml
# promptfooconfig.yaml
prompts:
  - "Classify this fraud complaint into a typology: {{narrative}}"
providers:
  - openai:gpt-4o-mini
  - openai:gpt-5-nano
tests:
  - vars:
      narrative: "Someone called claiming to be from the IRS, said I owed back taxes and would be arrested unless I paid immediately via gift cards."
    assert:
      - type: contains
        value: "government_impersonation_scam"
      - type: llm-rubric
        value: "Correctly identifies this as an impersonation-based scam, not identity theft"
```
This runs the same prompt against both models on the same test case, checking a deterministic assertion (`contains`) and a model-graded one (`llm-rubric`, same LLM-as-judge mechanism as DeepEval's `GEval`) — useful for comparing model/prompt versions side by side rather than evaluating one fixed pipeline.